# Transformer 2

## Mulit head Attention

먼저 트랜스포머 구조에서 'Multi head attention' 박스의 구조를 보자면,

  1. 입력에서 Q, K, V 생성                                                                                                                                    
  2. 여러 head로 나눔                                                                                                                                         
  3. 각 head가 attention 수행                                                                                                                                 
  4. head 결과들을 Concat                                                                                                                                     
  5. W_O로 선형변환                                                                                                                                           
  6. 그 결과가 Multi-Head Attention 블록의 출력이 됨                                                                                                          
  7. 그 다음에 Add & Norm으로 넘어감

## Mask

mask는 학습할 떄에만 적용되고 실제 기계 번역에서는 적용되지 않을까?
> No!!

디코더의 mask는 학습할 때도 쓰이고 실제 생성할 때도 필요하다

학습 때는 mask를 통해 디코더가 각 위치 자기 앞까지만 보게 한다

실제 번역(다음 단어 추론) 할 때에는 보통 디코더가 단어를 한 개씩 순차적으로 생성하는데, 

  1. 시작 토큰 입력                                                                                                                                           
  2. 첫 단어 생성                                                                                                                                             
  3. 지금까지 생성된 단어들을 넣어서 다음 단어 생성                                                                                                           
  4. 반복  

이 경우 아직 미래 단어 자체가 존재하지 않으니까, 추론 때에도 미래를 보지 않는 제약은 그대로 유지된다. 다만, 추론은 단어를 하나씩 생성하므로 미래 단어가 애초에 입력이 없기 때문에 원리적으로는 필요 없지만, 형식 상으로만 존재한다.  

> 추론에서도 디코더의 각 위치는 자기까지의 이전 토큰들만 반영해서 표현이 만들어진다. 
>
> mask는 학습 전용 전략이다. 이를 통해 RNN 처럼 단어를 하나씩 순서대로 처리하는 하지 않으면서도 한 번에 문장을 넣고 뒤의 단어를 mask한 것이다 -> 순차적인 RNN의 한계를 극복하면서도 정답을 미리 보는 반칙을 막음

## 인코더와 디코더의 embedding

인코더 - input emdedding  
디코더 - output emdedding 

각각 다른 임베딩이 들어간다. 

1. 인코더 임베딩의 정체
> 입력 언어의 단어들을 벡터로 만든다.  
> ex 한국어를 벡터로 만든다.

2. 디코더 임베딩의 정체
> 출력 언어의 단어들을 벡터로 만든다.  
> ex 영어를 벡터로 만든다.


**이때 디코더는 빙글빙글 돈다(자기회귀)**

트랜스포머는 문장을 한 번에 만들어서 뱉는게 아니라 하나 생성하고 다시 디코더에 입력으로 집어넣는 루프를 가진다. -> **디코더의 임베딩 정체가 바로 그것이다**

1회차: 디코더에 <sos>(시작 토큰)만 넣습니다. ->  결과: "I" 예측

2회차: 디코더에 [<sos>, "I"]를 넣습니다 -> 결과: "am" 예측

3회차: 디코더에 [<sos>, "I", "am"]을 넣습니다. -> 결과: "a" 예측

4회차: 디코더에 [<sos>, "I", "am", "a"]를 넣습니다. -> 결과: "student" 예측

[<sos>, "I", "am", "a", "student"]를 넣었더니 결과로 <eos>(종료 토큰)이 나오면 루프를 멈춥니다.

*물론, 이것은 실제 추론할 때의 과정이다. 학습할 때에는 모든 정보를 한 번에 밀어넣고(+ mask) 학습하기에 속도가 압도적으로 빠르다. 


**중요한 것은 빙글빙글 도는 것은 오직 디코더이다!!**
> 인코더는 한 번만 일한다. '나는 학생입니다'라는 문장을 받으면, 단 한번의 계산으로 모든 단어 사이의 관계를 파악해 정보 보따리 (K,V)를 만들어둔다
>
> 디코더: 인코더가 만들어 놓은 보따리를 매 회차 계속 들처보면서 다음에 올 단어를 하나씩 생성한다. 


## 인코더 디코더의 Attention (Encoder-Decoder Cross-Attention)

디코더가 i am 까지 단어를 만들었다고 가정해보자

디코더는 i am 다음에 올 단어를 찾기 위해 현재 자신의 상태를 Q로 만든다. 그리고 인코더가 준 K와 내적을 한다. -> 역서 attention socre를 만든다. -> 점수가 가장 높은 학생의 Attention Value를 비중있게 가져오게 된다!


Attention Score (어텐션 스코어): Query와 Key를 내적한 값입니다 ($Q \cdot K^T$).

Attention Value (어텐션 밸류): 우리가 흔히 말하는 **'Attention Value'**는 단순히 $V$ 자체를 의미하는 것이 아니라, Q와 K의 관계를 통해 **'가중치가 입혀진 V의 합'**을 의미

관련성 계산 Q dot K: 질문(Q)과 이름표(K)를 곱해 단어들끼리 얼마나 관련 있는지 점수를 낸다.

비중 결정 (Softmax): 그 점수를 확률(0~1 사이)로 바꿉니다. "이 단어는 80% 중요하고, 저 단어는 2% 중요해."

정보 추출 (V와 곱하기): 위에서 구한 확률을 실제 알맹이(V)에 곱합니다. 중요한 단어의 V는 크게 가져오고, 안 중요한 단어의 V는 작게 가져오는 것이다. 

최종 결과: 이렇게 계산된 값들을 모두 더한 것이 바로 다음 레이어로 전달될 최종 Attention Value가 된다. 

즉, attenion value는 문장 속 여러 단어들 중에서 현재 단어와 관련 있는 정보들만 필요한 만큼 쏙쏙 뽑아서 합쳐놓은 **농축된 문맥 벡터이다**

결국 멀트헤드셀프어텐션의 목적 그 복잡한 계산의 목적지는 "각 단어가 주변 맥락을 흡수하여 새롭게 태어난 결과물", 즉 Attention Value를 만드는 것이다!!

> 인코더의 존재 이유: "똑똑한 벡터 만들기"

인코더는 입력받은 단어들을 하나하나 뜯어본 뒤, 다음 층으로 넘겨줄 때 **"야, 내가 주변 다 살펴봤는데 이 단어는 여기서 이런 뜻이야"**라고 알려줘야 한다. 

이때 **'이런 뜻'**에 해당하는 실체가 바로 Attention Value이다. 

## 디코더 내부

디코더 내부 과정을 더 살펴보자. 현재 i am 까지 단어를 생성했다고 가정해보자. 지금 디코더에는 <sos>, i, am 이라는 데이터가 임베딩 되어 들어가 있느 ㄴ상태이다.


1. Masked Self-Attention: "내 안의 문맥 파악"가장 먼저 하는 일은 현재까지 내가 뱉은 단어들끼리 서로를 돌아보는 것입니다.

동작: "i"와 "am"이 서로 Q, K, V 연산을 합니다.결과: "am"이라는 단어는 "i"라는 주어와 연결되어 있다는 것을 깨닫습니다. "아, 내가 지금 '나'에 대해 설명하는 중이구나!"라는 내부 문맥이 담긴 벡터가 만들어집니다.

Masked인 이유: (학습 때만 해당하지만) 뒤에 올 "student"를 미리 보고 답을 베끼지 못하게 가려놓기 때문입니다.

2. Cross-Attention: "원본 문장에서 힌트 찾기"

여기서 "student"라는 정보가 외부(인코더)로부터 수입됩니다.


동작: 디코더의 "am"(Q)이 인코더가 준 "나는 학생이다"(K, V)를 훑습니다.결과: "내가 지금 'am'까지 말했는데, 인코더 문장에서 '나'랑 'am'에 대응되는 걸 빼면 남은 핵심 정보가 뭐지?"라고 물으니, 인코더의 **'학생'**이라는 단어 벡터가 강하게 응답합니다.

의미: 이 과정을 거치면 "am" 뒤에 올 내용이 인코더의 '학생'과 관련되어야 한다는 번역 힌트가 벡터에 합쳐집니다.

3. FFNN: "정보 다듬기 및 최종 판단"
위에서 얻은 "내부 문맥(i am)"과 "외부 힌트(학생)"가 섞인 복잡한 벡터를 다시 한번 정돈합니다.

동작: 각 단어 위치별로 고정된 신경망을 통과시킵니다.

결과: "좋아, 주어는 'i'고, 동사는 'am'이고, 인코더에서 찾아낸 핵심 키워드는 '학생'이야. 이걸 종합하면 다음에 올 가장 적절한 추상적 개념은 '학생'의 영어 형태겠군!"이라며 벡터를 최종적으로 보정합니다.

4. Linear & Softmax (마지막 관문)

사실 1, 2, 3번을 거쳐 나온 건 여전히 숫자로 가득한 벡터일 뿐입니다. 진짜 "student"라는 글자가 되는 건 디코더 블록 바로 위에 붙은 두 층 덕분입니다.

> Linear Layer: 디코더에서 나온 벡터를 우리 모델이 배운 전체 단어장(Vocabulary, 예: 5만 개 단어) 크기의 긴 벡터로 뻥튀기합니다.
>
> Softmax: 5만 개의 칸마다 확률을 채웁니다.
>
> i: 0.001%
>
> apple: 0.01%
>
> student: 85%
>
> teacher: 5%

결과: 확률이 가장 높은 **"student"**를 최종 선택해서 출력합니다!

정리해보면

- Masked Attention: "지금까지 'i am'까지 말했어." (내 상황 파악)

- Cross Attention: "인코더를 보니 '학생'이라는 정보가 남았네!" (정답 소스 확보)

- FFNN: "이걸 합치면 '학생' 명사가 와야겠군." (최종 추론)

- Softmax: "단어장 중 'student' 확률이 제일 높다, 이걸로 ouput을 내보내자!


-> 이렇게 [<sos>, i, am, student]가 디코더의 인풋으로 다시 들어갈 것이다(자기회귀시작)